In [ ]:
# 기존 연구 
import os
import json
from Src.dataset import load_annotations, get_vcd_images, encode_image_to_base64
from Src.vcd_decoding import generate_vcd_candidates

# 1. 경로 설정
ANNOTATION_PATH = "/Data/annotations/val.json"
IMAGE_DIR = "/Data/VizWiz_val/"
OUTPUT_FILE = "/Results/multiple_candidates_results.json"

# 2. 데이터 불러오기
val_data = load_annotations(ANNOTATION_PATH)
results_list = []

print(" VCD 기반 다중 후보 생성 파이프라인을 시작함")

# 테스트를 위해 [:5]만 실행할 것. 잘 되면 [:5]를 지우고 전체를 실행함.
for i, item in enumerate(val_data[:5]):
    image_filename = item['image']
    question = item['question']
    image_path = os.path.join(IMAGE_DIR, image_filename)

    if not os.path.exists(image_path):
        continue

    print(f"[{i+1}/{len(val_data)}] {image_filename} 처리 중")

    # Src/dataset.py의 기능을 사용하여 이미지 두 장(원본, 노이즈) 준비
    orig_img, noisy_img = get_vcd_images(image_path)
    orig_b64 = encode_image_to_base64(orig_img)
    noisy_b64 = encode_image_to_base64(noisy_img)

    # Src/vcd_decoding.py의 기능을 사용하여 LLaVA 답변 3개 추출
    candidates = generate_vcd_candidates(question, orig_b64, noisy_b64)

    # 결과 리스트에 추가
    results_list.append({
        "image": image_filename,
        "question": question,
        "candidates": candidates
    })

# 3. 결과 파일 저장 (Results 폴더)
os.makedirs("/Results", exist_ok=True)
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(results_list, f, ensure_ascii=False, indent=4)

print(f"\n 모든 후보 생성이 완료되었음. 저장 위치: {OUTPUT_FILE}")

In [ ]:
import os
import json

# Src 폴더의 함수들 불러오기 (실행 버튼을 대신 눌러줌.)
%run /Src/dataset.ipynb
%run /Src/vcd_decoding.ipynb

# 1. 경로 설정 (12시간 동안 돌린 02번 결과를 불러옴.)
BASELINE_FILE = "/Results/baseline_inference_results.json"
IMAGE_DIR = "/Data/VizWiz_val/"
OUTPUT_FILE = "/Results/multiple_candidates_results.json"

# 2. Baseline 결과 데이터 불러오기
try:
    with open(BASELINE_FILE, 'r', encoding='utf-8') as f:
        val_data = json.load(f)
    print(f" Baseline 데이터 로드 완료: 총 {len(val_data)}개")
except FileNotFoundError:
    print(" Baseline 파일을 찾을 수 없음. 경로를 확인해바람.")
    val_data = []

results_list = []

print(" B-VCD 도메인 특화 다중 후보 생성 파이프라인을 시작함")

# 일단 5개만 쾌속 테스트. (잘 되면 [:5]를 지우고 전체를 돌리세요)
for i, item in enumerate(val_data):
    image_filename = item['image']
    question = item['question']
    baseline_answer = item['answer'] # 12시간 돌린 귀중한 정답 (후보 1 재활용.)
    
    image_path = os.path.join(IMAGE_DIR, image_filename)

    if not os.path.exists(image_path):
        continue

    print(f"\n[{i+1}/{len(val_data)}] {image_filename} 처리 중")

    # Src/dataset.ipynb의 '물리 기반 확률적 센서 왜곡' 적용 (에러 났던 부분 수정 완료.)
    orig_img, domain_distorted_img = get_b_vcd_images(image_path, max_blur=30, brightness_factor=0.3, read_noise_std=5.0)
    
    orig_b64 = encode_image_to_base64(orig_img)
    noisy_b64 = encode_image_to_base64(domain_distorted_img)
    
    prompt = f"You are a helpful assistant for a visually impaired person. Please answer the question based on the image.\nQuestion: {question}"

    # --- 3가지 후보군 장전. ---
    # 후보 1 (Regular): API 호출 안 함. 기존 결과 재활용 (시간 엄청 절약 됨 )
    candidate_1 = baseline_answer
    
    # 후보 2 (Noisy): B-VCD 왜곡 이미지 + 낮은 온도
    print("  ->  후보 2 (B-VCD 노이즈) 추출 중")
    candidate_2 = generate_llava_response(prompt, noisy_b64, temperature=0.0)
    
    # 후보 3 (Diverse): 원본 이미지 + 높은 온도
    print("  ->  후보 3 (창의적 답변) 추출 중")
    candidate_3 = generate_llava_response(prompt, orig_b64, temperature=0.7)

    # 결과 리스트에 3가지 후보 모두 합쳐서 저장
    results_list.append({
        "image": image_filename,
        "question": question,
        "candidates": {
            "candidate_1_regular": candidate_1,
            "candidate_2_noisy": candidate_2,
            "candidate_3_diverse": candidate_3
        }
    })

# 3. 최종 다중 후보 파일 저장
os.makedirs("/Results", exist_ok=True)
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(results_list, f, ensure_ascii=False, indent=4)

print(f"\n 다중 후보 생성 완료. 저장 위치: {OUTPUT_FILE}")

✅ Baseline 데이터 로드 완료: 총 4319개
🚀 B-VCD 도메인 특화 다중 후보 생성 파이프라인을 시작합니다...

[1/4319] VizWiz_val_00000000.jpg 처리 중...
  -> 🤖 후보 2 (B-VCD 노이즈) 추출 중...
  -> 🤖 후보 3 (창의적 답변) 추출 중...

[2/4319] VizWiz_val_00000001.jpg 처리 중...
  -> 🤖 후보 2 (B-VCD 노이즈) 추출 중...
  -> 🤖 후보 3 (창의적 답변) 추출 중...

[3/4319] VizWiz_val_00000002.jpg 처리 중...
  -> 🤖 후보 2 (B-VCD 노이즈) 추출 중...
  -> 🤖 후보 3 (창의적 답변) 추출 중...

[4/4319] VizWiz_val_00000003.jpg 처리 중...
  -> 🤖 후보 2 (B-VCD 노이즈) 추출 중...
  -> 🤖 후보 3 (창의적 답변) 추출 중...

[5/4319] VizWiz_val_00000004.jpg 처리 중...
  -> 🤖 후보 2 (B-VCD 노이즈) 추출 중...
  -> 🤖 후보 3 (창의적 답변) 추출 중...

[6/4319] VizWiz_val_00000005.jpg 처리 중...
  -> 🤖 후보 2 (B-VCD 노이즈) 추출 중...
  -> 🤖 후보 3 (창의적 답변) 추출 중...

[7/4319] VizWiz_val_00000006.jpg 처리 중...
  -> 🤖 후보 2 (B-VCD 노이즈) 추출 중...
  -> 🤖 후보 3 (창의적 답변) 추출 중...

[8/4319] VizWiz_val_00000007.jpg 처리 중...
  -> 🤖 후보 2 (B-VCD 노이즈) 추출 중...
  -> 🤖 후보 3 (창의적 답변) 추출 중...

[9/4319] VizWiz_val_00000008.jpg 처리 중...
  -> 🤖 후보 2 (B-VCD 노이즈) 추출 중...
  -> 🤖 후보 3 (창의적 답변) 추출 중...

[1